# Gemma 4 — Persistent Chat History + Hybrid Routing (Steps 26-27)

Two infrastructure upgrades for production-quality local agents:

**Step 26 — Persistent chat history (SQLite)**  
Conversations survive restarts. Sessions are named and searchable. Full history
is available for context injection.

**Step 27 — Hybrid routing (local Gemma + cloud fallback)**  
Route easy/private queries to local Gemma 4. Route hard reasoning tasks to a
frontier cloud model. Uses `RouterQueryEngine` — the routing decision itself
stays local.

```bash
pip install llama-index-llms-ollama llama-index-llms-anthropic llama-index-core
```

## Step 26 — Persistent Chat History (SQLite)

In [ ]:
import sqlite3
import json
import datetime
from pathlib import Path
from typing import Optional


DB_PATH = "gemma4_chat_history.db"


def init_db(db_path: str = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS sessions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id INTEGER NOT NULL REFERENCES sessions(id),
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_messages_session ON messages(session_id)")
    conn.commit()
    return conn


conn = init_db()
print(f"Chat history DB initialised at {DB_PATH}")

In [ ]:
class ChatSession:
    """A named conversation that persists to SQLite."""

    def __init__(self, name: str, db_path: str = DB_PATH) -> None:
        self.name = name
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        now = datetime.datetime.utcnow().isoformat()
        self.conn.execute(
            "INSERT OR IGNORE INTO sessions (name, created_at, updated_at) VALUES (?, ?, ?)",
            (name, now, now),
        )
        self.conn.commit()
        row = self.conn.execute("SELECT id FROM sessions WHERE name = ?", (name,)).fetchone()
        self.session_id: int = row["id"]

    def add_message(self, role: str, content: str) -> None:
        now = datetime.datetime.utcnow().isoformat()
        self.conn.execute(
            "INSERT INTO messages (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
            (self.session_id, role, content, now),
        )
        self.conn.execute(
            "UPDATE sessions SET updated_at = ? WHERE id = ?", (now, self.session_id)
        )
        self.conn.commit()

    def get_history(self, limit: int = 20) -> list[dict]:
        rows = self.conn.execute(
            "SELECT role, content FROM messages WHERE session_id = ? ORDER BY id DESC LIMIT ?",
            (self.session_id, limit),
        ).fetchall()
        return [{"role": r["role"], "content": r["content"]} for r in reversed(rows)]

    def clear(self) -> None:
        self.conn.execute("DELETE FROM messages WHERE session_id = ?", (self.session_id,))
        self.conn.commit()
        print(f"Session '{self.name}' cleared.")

    @staticmethod
    def list_sessions(db_path: str = DB_PATH) -> list[dict]:
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            "SELECT s.name, s.updated_at, COUNT(m.id) as msg_count "
            "FROM sessions s LEFT JOIN messages m ON m.session_id = s.id "
            "GROUP BY s.id ORDER BY s.updated_at DESC"
        ).fetchall()
        return [dict(r) for r in rows]


print("ChatSession class ready.")

In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.core.llms import ChatMessage

llm = Ollama(model="gemma4:12b", request_timeout=120.0)


async def persistent_chat(session_name: str, user_input: str, system_prompt: str = "") -> str:
    """Chat with Gemma 4 using a named persistent session."""
    session = ChatSession(session_name)
    history = session.get_history(limit=20)

    messages = []
    if system_prompt:
        messages.append(ChatMessage(role="system", content=system_prompt))
    messages.extend([ChatMessage(role=m["role"], content=m["content"]) for m in history])
    messages.append(ChatMessage(role="user", content=user_input))

    response = await llm.achat(messages)
    reply = str(response.message.content)

    session.add_message("user", user_input)
    session.add_message("assistant", reply)
    return reply


# Example conversation
reply1 = await persistent_chat("crypto-research", "What is the difference between Ethereum and Base?")
print("Gemma:", reply1[:200], "...")

In [ ]:
# Continue the same session — Gemma remembers the context
reply2 = await persistent_chat(
    "crypto-research",
    "Which one has lower gas fees on average?",
)
print("Gemma:", reply2[:200], "...")

In [ ]:
# List all sessions
sessions = ChatSession.list_sessions()
print("Sessions:")
for s in sessions:
    print(f"  [{s['name']}] {s['msg_count']} messages — last active: {s['updated_at'][:10]}")

In [ ]:
# Review conversation history
session = ChatSession("crypto-research")
for msg in session.get_history():
    prefix = "You" if msg["role"] == "user" else "Gemma"
    print(f"[{prefix}] {msg['content'][:100]}")

## Step 27 — Hybrid Routing (Local Gemma + Cloud Fallback)

Route by query type:
- **Private / fast / simple** → local `gemma4:12b` via Ollama (stays on device)
- **Hard reasoning / long context** → `claude-haiku-4-5` or `claude-sonnet-4-6` (cloud)

The routing decision itself runs locally.

In [ ]:
import os

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

try:
    from llama_index.llms.anthropic import Anthropic
    cloud_llm = Anthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)
    print("Cloud fallback: claude-haiku-4-5 ready")
except ImportError:
    cloud_llm = None
    print("llama-index-llms-anthropic not installed. Install it for cloud fallback:")
    print("  pip install llama-index-llms-anthropic")

if not ANTHROPIC_API_KEY:
    print("Set ANTHROPIC_API_KEY for cloud routing to work.")

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

Settings.llm = llm  # routing decisions use the local model

# Build two minimal indices with contrasting content
private_docs = [
    Document(text="My Ethereum wallet address is 0xABCD. Balance: 2.5 ETH. Last tx: 2026-06-01."),
    Document(text="My Shopify store revenue for May 2026 was $4,200. Best seller: Midnight Pine candle."),
]
private_index = VectorStoreIndex.from_documents(private_docs)
private_engine = private_index.as_query_engine()

# Public knowledge index (would normally be larger)
public_docs = [
    Document(text="Ethereum is a decentralised smart contract platform using proof-of-stake."),
    Document(text="Base is an L2 built on the OP Stack, operated by Coinbase, with low fees."),
]
public_index = VectorStoreIndex.from_documents(public_docs)
public_engine = public_index.as_query_engine()

print("Indices built.")

In [ ]:
# Hybrid router: local Gemma chooses between local-private and local-public engines
# For truly hard tasks you'd add a cloud engine tool here
router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(llm=llm),
    query_engine_tools=[
        QueryEngineTool.from_defaults(
            query_engine=private_engine,
            description="Use for questions about MY personal portfolio, wallet, store revenue, or private data.",
        ),
        QueryEngineTool.from_defaults(
            query_engine=public_engine,
            description="Use for general questions about crypto, blockchain, or public technical topics.",
        ),
    ],
)

# Private query → routes to private engine
r1 = router_engine.query("What is my current ETH balance?")
print("Private query:", r1)

# Public query → routes to public engine
r2 = router_engine.query("What consensus mechanism does Ethereum use?")
print("Public query:", r2)

In [ ]:
# Full hybrid: local Gemma for private/fast, cloud for complex reasoning
async def hybrid_chat(
    user_input: str,
    is_private: bool = False,
    force_cloud: bool = False,
) -> str:
    """Route to local or cloud LLM based on sensitivity and complexity."""
    # Private data always stays local
    if is_private or force_cloud is False and len(user_input) < 200:
        chosen_llm = llm
        route = "local (Gemma 4)"
    elif cloud_llm is not None and ANTHROPIC_API_KEY:
        chosen_llm = cloud_llm
        route = "cloud (Claude Haiku)"
    else:
        chosen_llm = llm
        route = "local (Gemma 4, cloud not available)"

    print(f"→ Routing to: {route}")
    response = await chosen_llm.acomplete(user_input)
    return str(response)


# Short/private → local
r = await hybrid_chat("What is 2+2?", is_private=False)
print(r[:100])

# Private data → always local regardless
r2 = await hybrid_chat("Summarise my wallet activity", is_private=True)
print(r2[:100])

## Step 28 — Local Embedding Benchmark

Compare `BAAI/bge-small-en-v1.5` vs `nomic-embed-text` for RAG retrieval quality
on your own documents.

In [ ]:
import time
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


def benchmark_embeddings(texts: list[str], model_name: str) -> dict:
    embed_model = HuggingFaceEmbedding(model_name=model_name)
    start = time.perf_counter()
    embeddings = embed_model.get_text_embedding_batch(texts)
    elapsed = time.perf_counter() - start
    return {
        "model": model_name,
        "dim": len(embeddings[0]),
        "texts": len(texts),
        "time_s": round(elapsed, 3),
        "time_per_text_ms": round(elapsed / len(texts) * 1000, 2),
    }


sample_texts = [
    "Ethereum uses proof-of-stake consensus since the Merge in 2022.",
    "My grocery receipt from Trader Joe's totalled $87.43 last Tuesday.",
    "LlamaIndex provides a framework for building RAG applications with local and cloud LLMs.",
    "Gemma 4 is Google's multimodal open model with 128K context window.",
    "The candle store had $4,200 revenue in May with the pine scent as the bestseller.",
]

models_to_test = [
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
]

results = []
for model in models_to_test:
    print(f"Benchmarking {model}...")
    try:
        result = benchmark_embeddings(sample_texts, model)
        results.append(result)
        print(f"  dim={result['dim']}, {result['time_per_text_ms']}ms/text")
    except Exception as e:
        print(f"  Failed: {e}")

print("\nBenchmark summary:")
for r in results:
    print(f"  {r['model']}: {r['dim']}d, {r['time_per_text_ms']}ms/text")

In [ ]:
# Retrieval quality comparison — same query, both models
from llama_index.core import Settings

TEST_QUERY = "What consensus mechanism does Ethereum use?"
TEST_DOCS = [
    Document(text="Ethereum uses proof-of-stake after the Merge in September 2022."),
    Document(text="Bitcoin uses proof-of-work with SHA-256 hashing."),
    Document(text="Solana uses proof-of-history combined with proof-of-stake."),
    Document(text="My ETH balance is 2.5 coins in my cold wallet."),
]

for model_name in ["BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5"]:
    try:
        Settings.embed_model = HuggingFaceEmbedding(model_name=model_name)
        idx = VectorStoreIndex.from_documents(TEST_DOCS)
        retriever = idx.as_retriever(similarity_top_k=2)
        nodes = retriever.retrieve(TEST_QUERY)
        print(f"\n{model_name}:")
        for n in nodes:
            print(f"  [{n.score:.3f}] {n.text[:80]}")
    except Exception as e:
        print(f"  {model_name} failed: {e}")